# 01 — Validate Snowflake mirror
Attach the mirrored database (or a Lakehouse containing shortcuts to it) before running. The notebook is read-only and fails fast on missing tables, null business keys, or duplicate declared keys.

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

expected = {
    'aircraft': ['tail_number'], 'route': ['route_id'],
    'flight': ['flight_no', 'date'], 'booking': ['booking_ref'],
    'customer': ['customer_id'], 'loyalty_transaction': [],
    'cargo_shipment': ['awb_number'], 'maintenance_event': [],
    'crew_roster': [], 'disruption_event': []
}
available = {r.tableName.lower(): r.tableName for r in spark.sql('SHOW TABLES').collect()}
missing = sorted(set(expected) - set(available))
assert not missing, f'Missing mirrored tables: {missing}'
print(f'Validated table presence at {datetime.now(timezone.utc).isoformat()}')

In [ ]:
results = []
failures = []
for logical, keys in expected.items():
    df = spark.table(available[logical])
    cols = {c.lower(): c for c in df.columns}
    count = df.count()
    null_keys = 0
    duplicates = 0
    if keys:
        absent = [k for k in keys if k not in cols]
        if absent:
            failures.append(f'{logical}: missing key columns {absent}')
        else:
            actual = [cols[k] for k in keys]
            null_keys = df.where(F.greatest(*[F.col(c).isNull().cast('int') for c in actual]) == 1).count()
            duplicates = df.groupBy(*actual).count().where('count > 1').count()
            if null_keys or duplicates:
                failures.append(f'{logical}: null_keys={null_keys}, duplicate_keys={duplicates}')
    date_cols = [cols[c] for c in ('date', 'event_date', 'duty_date', 'joined_date') if c in cols]
    max_date = df.agg(F.max(date_cols[0])).first()[0] if date_cols else None
    results.append((logical, count, null_keys, duplicates, str(max_date) if max_date else None))

display(spark.createDataFrame(results, ['table', 'rows', 'null_business_keys', 'duplicate_business_keys', 'max_business_date']))
assert not failures, 'Mirror validation failed:\n' + '\n'.join(failures)
print('Mirror validation passed.')